In [1]:
# Cell 1 — Imports and paths

import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

GRAPH_FILE = Path(
    "D:/adaptive_federated_gnn/data/processed/edge_graph.pt"
)

CLIENT_FILE = Path(
    "D:/adaptive_federated_gnn/outputs/day10/federated_clients.pt"
)

DAY12_OUTPUT = Path(
    "D:/adaptive_federated_gnn/outputs/day12"
)

DAY12_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

Device: cpu


In [2]:
# Cell 2 — Load graph

graph_data = torch.load(
    GRAPH_FILE,
    map_location="cpu",
    weights_only=False
)

edge_index = graph_data["edge_index"].long()

if "edge_features" in graph_data:
    edge_feature_size = graph_data["edge_features"].shape[1]

elif "edge_attr" in graph_data:
    edge_feature_size = graph_data["edge_attr"].shape[1]

else:
    raise KeyError(
        f"No edge features found. Available keys: {list(graph_data.keys())}"
    )

num_nodes = int(
    edge_index.max().item()
) + 1

print("Number of nodes:", num_nodes)
print("Number of graph edges:", edge_index.shape[1])
print("Edge feature size:", edge_feature_size)

Number of nodes: 4404
Number of graph edges: 286095
Edge feature size: 67


In [3]:
# Cell 3 — Load federated clients

clients = torch.load(
    CLIENT_FILE,
    map_location="cpu",
    weights_only=False
)

print("Clients:")

for client_name, client_data in clients.items():

    print(
        client_name,
        "->",
        client_data["edge_features"].shape[0],
        "edges"
    )

Clients:
client_1 -> 66755 edges
client_2 -> 66755 edges
client_3 -> 66756 edges


In [4]:
# Cell 4 — Define EdgeGCN

import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class EdgeGCN(nn.Module):

    def __init__(
        self,
        num_nodes,
        edge_feature_size,
        embedding_size=16,
        hidden_size=32
    ):
        super().__init__()

        self.node_embeddings = nn.Embedding(
            num_nodes,
            embedding_size
        )

        self.conv1 = GCNConv(
            embedding_size,
            hidden_size
        )

        self.conv2 = GCNConv(
            hidden_size,
            hidden_size
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                hidden_size * 2 + edge_feature_size,
                64
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(
        self,
        edge_index,
        edge_features
    ):

        node_ids = torch.arange(
            self.node_embeddings.num_embeddings,
            device=edge_index.device
        )

        x = self.node_embeddings(node_ids)

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        source_nodes = edge_index[0]
        destination_nodes = edge_index[1]

        source_embeddings = x[source_nodes]
        destination_embeddings = x[destination_nodes]

        combined_features = torch.cat(
            [
                source_embeddings,
                destination_embeddings,
                edge_features
            ],
            dim=1
        )

        return self.classifier(
            combined_features
        )


print("EdgeGCN defined successfully.")

c:\Users\daksh\anaconda3\envs\adaptive_gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EdgeGCN defined successfully.


In [5]:
# Cell 5 — Load local client models

client_models = {}

for client_name in clients.keys():

    model = EdgeGCN(
        num_nodes=num_nodes,
        edge_feature_size=edge_feature_size
    )

    model.load_state_dict(
        torch.load(
            f"D:/adaptive_federated_gnn/outputs/day11/{client_name}_model.pt",
            map_location="cpu",
            weights_only=True
        )
    )

    model.eval()

    client_models[client_name] = model

    print(
        client_name,
        "model loaded successfully"
    )

client_1 model loaded successfully
client_2 model loaded successfully
client_3 model loaded successfully


FedAvg

The basic FedAvg formula is:

$$ W_{global} = \sum_{k=1}^{K} \frac{n_k}{N}W_k $$

where:

\(W_k\) = model parameters of client \(k\)
\(n_k\) = number of training edges on client \(k\)
\(N\) = total training edges
\(W_{global}\) = new global model

Because our clients have almost equal numbers of edges, their weights will be almost

In [6]:
# Cell 6 — Calculate FedAvg client weights

client_sizes = {}

for client_name, client_data in clients.items():

    client_sizes[client_name] = (
        client_data["edge_features"].shape[0]
    )

total_edges = sum(
    client_sizes.values()
)

client_weights = {
    client_name:
    client_sizes[client_name] / total_edges
    for client_name in client_sizes
}

print("Client weights:")

for client_name, weight in client_weights.items():

    print(
        client_name,
        "->",
        weight
    )

print(
    "\nWeight sum:",
    sum(client_weights.values())
)

Client weights:
client_1 -> 0.33333166888038906
client_2 -> 0.33333166888038906
client_3 -> 0.3333366622392218

Weight sum: 1.0


In [7]:
# Cell 7 — FedAvg aggregation function

def fedavg(
    client_models,
    client_weights
):

    global_model = EdgeGCN(
        num_nodes=num_nodes,
        edge_feature_size=edge_feature_size
    )

    global_state = global_model.state_dict()

    for parameter_name in global_state:

        aggregated_parameter = torch.zeros_like(
            global_state[parameter_name]
        )

        for client_name, client_model in client_models.items():

            client_parameter = (
                client_model.state_dict()[parameter_name]
            )

            weight = client_weights[client_name]

            aggregated_parameter += (
                weight * client_parameter
            )

        global_state[parameter_name] = (
            aggregated_parameter
        )

    global_model.load_state_dict(
        global_state
    )

    return global_model


print("FedAvg function defined successfully.")

FedAvg function defined successfully.


In [8]:
# Cell 8 — Perform FedAvg

global_model = fedavg(
    client_models,
    client_weights
)

global_model.eval()

print("Global model created successfully.")

Global model created successfully.


In [9]:
# Cell 9 — Save global model

GLOBAL_MODEL_FILE = (
    DAY12_OUTPUT / "global_model_round1.pt"
)

torch.save(
    global_model.state_dict(),
    GLOBAL_MODEL_FILE
)

print(
    "Global model saved to:",
    GLOBAL_MODEL_FILE
)

Global model saved to: D:\adaptive_federated_gnn\outputs\day12\global_model_round1.pt


In [10]:
# Cell 10 — Verify global model parameters

global_model_loaded = EdgeGCN(
    num_nodes=num_nodes,
    edge_feature_size=edge_feature_size
)

global_model_loaded.load_state_dict(
    torch.load(
        GLOBAL_MODEL_FILE,
        map_location="cpu",
        weights_only=True
    )
)

global_model_loaded.eval()

print("Global model loaded successfully.")

total_parameters = sum(
    parameter.numel()
    for parameter in global_model_loaded.parameters()
)

print(
    "Total model parameters:",
    total_parameters
)

Global model loaded successfully.
Total model parameters: 80642


In [11]:
# Cell 11 — Save FedAvg configuration

federated_config = {
    "algorithm": "FedAvg",
    "num_clients": len(clients),
    "client_sizes": client_sizes,
    "client_weights": client_weights,
    "num_nodes": num_nodes,
    "edge_feature_size": edge_feature_size,
    "model": "EdgeGCN",
    "round": 1
}

torch.save(
    federated_config,
    DAY12_OUTPUT / "fedavg_config_round1.pt"
)

print("FedAvg configuration saved.")

FedAvg configuration saved.


In [12]:
# Cell 12 — Save FedAvg summary

summary_rows = []

for client_name in clients.keys():

    summary_rows.append({
        "client": client_name,
        "num_edges": client_sizes[client_name],
        "fedavg_weight": client_weights[client_name]
    })

summary_rows.append({
    "client": "total",
    "num_edges": total_edges,
    "fedavg_weight": sum(client_weights.values())
})

fedavg_summary = pd.DataFrame(
    summary_rows
)

fedavg_summary.to_csv(
    DAY12_OUTPUT / "fedavg_client_weights.csv",
    index=False
)

display(fedavg_summary)

,client,num_edges,fedavg_weight
0,client_1,66755,0.333332
1,client_2,66755,0.333332
2,client_3,66756,0.333337
3,total,200266,1.000000


In [14]:
# Cell 13 — Day 12 completion check

print("DAY 12 — FEDAVG COMPLETION CHECK")


files_to_check = [
    "global_model_round1.pt",
    "fedavg_config_round1.pt",
    "fedavg_client_weights.csv"
]

for file_name in files_to_check:

    file_path = DAY12_OUTPUT / file_name

    print(
        file_name,
        "->",
        "OK" if file_path.exists() else "MISSING"
    )

print("\nFedAvg client weights:")

display(fedavg_summary)

DAY 12 — FEDAVG COMPLETION CHECK
global_model_round1.pt -> OK
fedavg_config_round1.pt -> OK
fedavg_client_weights.csv -> OK

FedAvg client weights:


,client,num_edges,fedavg_weight
0,client_1,66755,0.333332
1,client_2,66755,0.333332
2,client_3,66756,0.333337
3,total,200266,1.000000
